In [18]:
import os

IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IS_KAGGLE:
    !git clone https://github.com/williamalxndr/mcts-engine.git
    %cd mcts-engine

In [19]:
!pip install -r requirements.txt

zsh:1: command not found: pip


In [20]:
if IS_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

HF_TOKEN set: False


In [22]:
import torch

n_gpus = torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

config_path = "configs/test.yaml"

!cat {config_path}

GPUs available: 0
game: chess
version: V2
file_name: test_checkpoint

mcts:
  num_rollout: 32
  mcts_batch_size: 4

training:
  duration: null
  iterations: 3
  steps_per_iter: 5
  train_batch_size: 8
  replay_buffer_max_size: 1000
  num_selfplay: 1

logging:
  kaggle: false
  verbose: true
  log_interval: 1

hf:
  push_to_hf: false
  load_from_hf: false
  repo_id: null

In [23]:
if n_gpus >= 1:
    cmd = f"torchrun --nproc_per_node={n_gpus} -m training.pipeline --config {config_path}"
else:
    cmd = f"python3 -m training.pipeline --config {config_path}"

print(cmd)
!{cmd}

python3 -m training.pipeline --config configs/test.yaml
Network loaded from local checkpoint checkpoints/chess/V2/test_checkpoint.pt
Buffer loaded from checkpoints
loss: 6.1623 | policy loss: 4.4123 | value loss: 1.7500 ━━━━━━━━━━ 3/3 • 0:01:44• 0:01:44
Network saved at checkpoints/chess/V2/test_checkpoint.pt

Training finished after 0h 1m 44s! To play against the trained network, run:
  python3 -m arena.play --game chess --version V2 --file_name test_checkpoint
  OR
  python3 -m arena.play --path checkpoints/chess/V2/test_checkpoint.pt
